# Cambridge Police: Clean Crash Records

Fetches the quarterly updated, more detailed Cambridge Police Department crash dataset from Cambridge Open Data, keeps records from 2021 onward, and normalizes them to the Mass Crash Map columns.

Output: `cambridge-police-crashes.csv` in the notebook's working directory. This is a standalone export; it does not merge or deduplicate against MassDOT.

## Data and interpretation

The source is the README's [Cambridge Police Department Crash Data - Updated](https://data.cambridgema.gov/Public-Safety/Police-Department-Crash-Data-Updated/gb5w-yva3) dataset (Socrata ID `gb5w-yva3`), not the daily, less detailed CPD Crash Log. Injury severity is 1 for a fatal injury, 2 for other reported injuries, and 0 when no injury is reported. `p1_non_motorist_desc` and `p2_non_motorist_desc` provide vulnerable-user classifications; CPD's calculated cyclist and pedestrian flags are included as a fallback.

The source does not provide a public crash identifier, so IDs are generated as `CPD_<download row number>` and can change when the dataset changes. CPD notes that a `12:00 AM` time can mean the time is unknown. Coordinates are copied only when present in the source's geocoded `location`; missing coordinates are not geocoded here.

In [1]:
%pip install pandas requests

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path

import pandas as pd
import requests

DATA_URL = "https://data.cambridgema.gov/resource/gb5w-yva3.json"
START_YEAR = 2021
PAGE_SIZE = 5000
REQUEST_TIMEOUT = 60
OUTPUT_FILE = Path("cambridge-police-crashes.csv")

## Retrieve the source table

The helper fetches every page in date order and raises an error for an unsuccessful HTTP response or an unexpected response shape.

In [3]:
def fetch_dataset() -> pd.DataFrame:
    records = []
    offset = 0

    while True:
        response = requests.get(
            DATA_URL,
            params={
                "$limit": PAGE_SIZE,
                "$offset": offset,
                "$order": "date_time ASC",
            },
            timeout=REQUEST_TIMEOUT,
        )
        response.raise_for_status()
        page = response.json()
        if not isinstance(page, list):
            raise RuntimeError("Socrata returned an unexpected response format")

        records.extend(page)
        if len(page) < PAGE_SIZE:
            break
        offset += PAGE_SIZE

    return pd.DataFrame.from_records(records)

In [4]:
crash_raw = fetch_dataset()
print(f"CPD crash records retrieved: {len(crash_raw):,}")
print(f"Available source columns: {len(crash_raw.columns):,}")
display(crash_raw.head())

CPD crash records retrieved: 17,084
Available source columns: 159


,date_time,day_of_week,object_1,object_2,street_number,street_name,cross_street,near_street,location,may_involve_cyclist,...,p2_non_motorist_desc,p2_non_motorist_action,p2_non_motorist_location,p2_safety_equipment,p2_age,p2_sex,p2_seat_position,p2_safety_system,p2_trapped,p2_veh_owner
0,2015-01-01T00:45:00.000,Thursday,PASSENGER CAR ...,PASSENGER CAR ...,,GREEN ST. ...,PLEASANT ST. ...,,"{'human_address': '{""address"": ""GREEN ST"", ""ci...",,...,,,,,,MALE ...,"FRONT SEAT - LEFT SIDE, OR MOTORCYLE DRIVER ...",SHOULDER AND LAP BELT USED ...,NOT TRAPPED ...,Y
1,2015-01-01T01:12:00.000,Thursday,PASSENGER CAR ...,PASSENGER CAR ...,,MASSACHUSETTS AVENUE ...,LANDSDOWNE STREET ...,,"{'human_address': '{""address"": ""MASSACHUSETTS ...",,...,,,,,,FEMALE ...,FRONT SEAT - RIGHT SIDE ...,UNKNOWN ...,NOT TRAPPED ...,
2,2015-01-01T01:54:00.000,Thursday,PASSENGER CAR ...,,,LOCKE ST. ...,SHEA RD. ...,,"{'human_address': '{""address"": ""LOCKE ST"", ""ci...",,...,,,,,,,,,,
3,2015-01-01T04:14:00.000,Thursday,PASSENGER CAR ...,PASSENGER CAR ...,225,CAMBRIDGE STREET ...,,THIRD STREET,"{'latitude': '42.371', 'longitude': '-71.07969...",,...,,,,,,MALE ...,FRONT SEAT - RIGHT SIDE ...,UNKNOWN ...,NOT TRAPPED ...,
4,2015-01-01T06:52:00.000,Thursday,PASSENGER CAR ...,PASSENGER CAR ...,,BROADWAY ST. ...,GALILEO BLVD. ...,,"{'human_address': '{""address"": ""BROADWAY ST"", ...",,...,,,,,,FEMALE ...,"FRONT SEAT - LEFT SIDE, OR MOTORCYLE DRIVER ...",SHOULDER AND LAP BELT USED ...,NOT TRAPPED ...,Y


Rows without a parseable timestamp or earlier than 2021 are excluded. Injury descriptions set severity, non-motorist descriptions set user flags, and coordinates are extracted from the Socrata location object when available. The source `street_name` is retained in the output after trimming padding; `interstate` is omitted because this CPD dataset does not report it.

In [ ]:
OUTPUT_COLUMNS = [
    "source", "id", "muni", "year", "date", "severity",
    "time", "police", "cyclist", "pedestrian", "other",
    "lat", "lng", "street_name", "datetime",
]
REQUIRED_COLUMNS = {
    "date_time", "p1_injury", "p2_injury", "street_name",
    "p1_non_motorist_desc", "p2_non_motorist_desc",
}
missing_columns = REQUIRED_COLUMNS.difference(crash_raw.columns)
if missing_columns:
    raise ValueError(f"CPD data is missing required columns: {sorted(missing_columns)}")

parsed_timestamp = pd.to_datetime(crash_raw["date_time"], errors="coerce")
keep = parsed_timestamp.notna() & parsed_timestamp.dt.year.ge(START_YEAR)
records = crash_raw.loc[keep].copy()
event_time = parsed_timestamp.loc[keep]

injury_descriptions = records[["p1_injury", "p2_injury"]].fillna("").astype("string")
injury_descriptions = injury_descriptions.apply(lambda column: column.str.strip().str.upper())
fatal = injury_descriptions.eq("FATAL INJURY").any(axis=1)
reported_injury = injury_descriptions.apply(
    lambda column: column.str.contains(r"\bINJURY\b", regex=True, na=False)
    & ~column.isin(["NO INJURY", "NO APPARENT INJURY", "DECEASED - NOT CAUSED BY CRASH"])
).any(axis=1)
severity = pd.Series(0, index=records.index, dtype="int8")
severity.loc[reported_injury] = 2
severity.loc[fatal] = 1

non_motorist = records[["p1_non_motorist_desc", "p2_non_motorist_desc"]].fillna("").astype("string")
non_motorist = non_motorist.apply(lambda column: column.str.strip().str.upper())
non_motorist_text = non_motorist.agg(" ".join, axis=1)

def has_pattern(values: pd.Series, pattern: str) -> pd.Series:
    return values.str.contains(pattern, case=False, regex=True, na=False)

cyclist = has_pattern(
    non_motorist_text,
    r"\b(?:CYCLIST|BICYCLIST|BICYCLE|HAND CYCLIST|TRICYCLIST|SKATER|SKATEBOARDER|NON[- ]MOTORIZED SCOOTER|OTHER MICROMOBILITY)\b",
)
pedestrian = has_pattern(
    non_motorist_text,
    r"\b(?:PEDESTRIAN|NON[- ]MOTORIZED WHEELCHAIR|ELECTRIC PERSONAL ASSISTIVE MOBILITY DEVICE|EMERGENCY RESPONDER|ROADWAY WORKER|UTILITY WORKER)\b",
)
other_user = has_pattern(
    non_motorist_text,
    r"\b(?:MOTORIZED BICYCLIST|MOTORIZED SCOOTER|MOPED|OTHER NON[- ]MOTORIST|OTHER VULNERABLE USER|TRAIN|TROLLEY|FARM EQUIPMENT|UNKNOWN)\b",
)

if "may_involve_cyclist" in records.columns:
    cyclist |= records["may_involve_cyclist"].astype("string").str.strip().eq("1")
if "may_involve_pedestrian" in records.columns:
    pedestrian |= records["may_involve_pedestrian"].astype("string").str.strip().eq("1")

event_columns = [
    column for column in ["first_harmful_event", "v1_most_harmful_event", "v2_most_harmful_event"]
    if column in records.columns
]
if event_columns:
    event_text = records[event_columns].fillna("").astype("string").agg(" ".join, axis=1)
    other_user |= has_pattern(
        event_text,
        r"\b(?:MOPED|MOTORIZED SCOOTER|MOTORIZED BICYCLIST|OTHER VULNERABLE USER|TRAIN|TROLLEY|FARM EQUIPMENT)\b",
    )

locations = records["location"] if "location" in records.columns else pd.Series(index=records.index, dtype="object")
latitude = locations.map(lambda value: value.get("latitude") if isinstance(value, dict) else pd.NA)
longitude = locations.map(lambda value: value.get("longitude") if isinstance(value, dict) else pd.NA)
street_name = records["street_name"].fillna("").astype("string").str.strip()

cambridge_clean = pd.DataFrame(index=records.index)
cambridge_clean["source"] = "CambridgePD"
cambridge_clean["id"] = "CPD_" + records.index.astype(str)
cambridge_clean["muni"] = "CAMBRIDGE"
cambridge_clean["year"] = event_time.dt.year.astype("Int64")
cambridge_clean["date"] = event_time.dt.strftime("%Y-%m-%d")
cambridge_clean["severity"] = severity
cambridge_clean["time"] = event_time.dt.strftime("%H:%M:%S")
cambridge_clean["police"] = pd.NA
cambridge_clean["cyclist"] = cyclist.astype("int8")
cambridge_clean["pedestrian"] = pedestrian.astype("int8")
cambridge_clean["other"] = other_user.astype("int8")
cambridge_clean["lat"] = pd.to_numeric(latitude, errors="coerce")
cambridge_clean["lng"] = pd.to_numeric(longitude, errors="coerce")
cambridge_clean["street_name"] = street_name
cambridge_clean["datetime"] = event_time.dt.strftime("%Y-%m-%d %H:%M:%S")
cambridge_clean = cambridge_clean[OUTPUT_COLUMNS]
cambridge_clean = cambridge_clean.sort_values("datetime").reset_index(drop=True)

assert cambridge_clean.columns.tolist() == OUTPUT_COLUMNS
assert cambridge_clean["id"].is_unique
print(f"Clean CPD records from {START_YEAR} onward: {len(cambridge_clean):,}")
print(cambridge_clean["severity"].value_counts(dropna=False).sort_index())
print("Records with coordinates:", cambridge_clean[["lat", "lng"]].notna().all(axis=1).sum())
display(cambridge_clean.head())

Clean CPD records from 2021 onward: 8,575
severity
0    7037
1       4
2    1534
Name: count, dtype: int64
Records with coordinates: 4822


,source,id,muni,year,date,severity,time,police,cyclist,pedestrian,other,lat,lng,interstate,datetime
0,CambridgePD,CPD_8509,CAMBRIDGE,2021,2021-01-03,0,17:51:00,<NA>,0,0,0,42.37047,-71.07597,<NA>,2021-01-03 17:51:00
1,CambridgePD,CPD_8510,CAMBRIDGE,2021,2021-01-04,2,12:30:00,<NA>,0,0,0,42.37327,-71.09726,<NA>,2021-01-04 12:30:00
2,CambridgePD,CPD_8511,CAMBRIDGE,2021,2021-01-04,0,12:36:00,<NA>,0,0,0,NaN,NaN,<NA>,2021-01-04 12:36:00
3,CambridgePD,CPD_8512,CAMBRIDGE,2021,2021-01-04,0,16:56:00,<NA>,0,0,0,42.36528,-71.08737,<NA>,2021-01-04 16:56:00
4,CambridgePD,CPD_8513,CAMBRIDGE,2021,2021-01-04,0,18:54:00,<NA>,0,0,0,NaN,NaN,<NA>,2021-01-04 18:54:00


In [6]:
cambridge_clean.to_csv(OUTPUT_FILE, index=False)
print(f"Saved {len(cambridge_clean):,} records to {OUTPUT_FILE.resolve()}")

Saved 8,575 records to C:\Users\johnbest\mass-crash-map\data\cambridge-police-crashes.csv
